In [3]:
# Import necessary libraries
import os
import time
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.feature_selection import RFE
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# Directory containing datasets
input_folder = "dataset/"  # Replace with your folder path
output_csv = "modelXGBoost_RFE.csv"

# Initialize a list to store results
results = []

# Loop through all files in the folder
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):  # Process only CSV files
        dataset_path = os.path.join(input_folder, filename)
        
        # Start time for processing the dataset
        start_time = time.time()

        # Load dataset
        print(f"Processing {filename}...")
        df = pd.read_csv(dataset_path)

        # Separate features (X) and target (y)
        X = df.iloc[:, :-1]
        y = df.iloc[:, -1]
        
        # Encode non-numeric columns
        for col in X.select_dtypes(include=['object']).columns:
            X[col] = pd.factorize(X[col])[0]
        
        # Encode target if it is categorical
        if y.dtype == 'object':
            y = pd.factorize(y)[0]

        # Map -1 to 0 to ensure binary class labels are 0 and 1
        y = np.where(y == -1, 0, y)
        
        # Handle imbalance in target using SMOTE
        smote = SMOTE(random_state=42, k_neighbors=3)
        X_resampled, y_resampled = smote.fit_resample(X, y)

        # Split into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(
            X_resampled, y_resampled, test_size=0.2, random_state=42
        )

        # Feature selection using RFE with XGBoost
        xgb_for_rfe = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
        rfe = RFE(estimator=xgb_for_rfe, n_features_to_select=10)  # Adjust number of features to select
        rfe.fit(X_train, y_train)

        # Get selected features
        selected_features = X.columns[rfe.support_]
        X_train_selected = rfe.transform(X_train)
        X_test_selected = rfe.transform(X_test)

        # Train final XGBoost model on selected features
        xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
        xgb_model.fit(X_train_selected, y_train)

        # Evaluate the model
        y_pred = xgb_model.predict(X_test_selected)
        y_proba = xgb_model.predict_proba(X_test_selected)[:, 1]
        
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        roc_auc = roc_auc_score(y_test, y_proba)
        
        # Confusion matrix to calculate false positive rate
        conf_matrix = confusion_matrix(y_test, y_pred)
        false_positive_rate = conf_matrix[0, 1] / conf_matrix[0].sum() if conf_matrix[0].sum() > 0 else 0

        # End time for processing the dataset
        end_time = time.time()
        runtime = end_time - start_time

        # Save results
        results.append({
            "Dataset Name": filename,
            "Number of Rows": df.shape[0],
            "Number of Columns": df.shape[1],
            "Number of Selected Features": len(selected_features),
            "Selected Features": ', '.join(selected_features),
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "False Positive Rate": false_positive_rate,
            "ROC AUC": roc_auc,
            "Runtime (seconds)": runtime
        })

# Save results to a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)

print(f"Results saved to {output_csv}")


Processing ds_100K20_.csv...


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:34:50] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:34:51] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:34:52] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, Us

Processing ds_10K18.csv...


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:34:57] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:34:58] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:34:58] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, Us

Processing ds_10K50_rev.csv...


...

Processing ds_11K89.csv...


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\joblib\externals\loky\backend\context.py", line 282, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:35:20] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgbo

Processing ds_129K112.csv...


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:36:00] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:36:02] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:36:03] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, Us

MemoryError: Unable to allocate 94.7 MiB for an array with shape (100, 124073) and data type float64

# Optimze Memory

In [4]:
import os
import time
import gc  # Import garbage collection module
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.feature_selection import RFE
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# Directory containing datasets
input_folder = "dataset/"  # Replace with your folder path
output_csv = "modelXGBoost_RFE.csv"

# Initialize a list to store results
results = []

# Loop through all files in the folder
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):  # Process only CSV files
        dataset_path = os.path.join(input_folder, filename)
        
        # Start time for processing the dataset
        start_time = time.time()

        try:
            # Load dataset
            print(f"Processing {filename}...")
            df = pd.read_csv(dataset_path)

            # Separate features (X) and target (y)
            X = df.iloc[:, :-1]
            y = df.iloc[:, -1]
            
            # Encode non-numeric columns
            for col in X.select_dtypes(include=['object']).columns:
                X[col] = pd.factorize(X[col])[0]
            
            # Encode target if it is categorical
            if y.dtype == 'object':
                y = pd.factorize(y)[0]

            # Map -1 to 0 to ensure binary class labels are 0 and 1
            y = np.where(y == -1, 0, y)
            
            # Handle imbalance in target using SMOTE
            smote = SMOTE(random_state=42, k_neighbors=3)
            X_resampled, y_resampled = smote.fit_resample(X, y)

            # Split into training and testing sets
            X_train, X_test, y_train, y_test = train_test_split(
                X_resampled, y_resampled, test_size=0.2, random_state=42
            )

            # Feature selection using RFE with XGBoost
            xgb_for_rfe = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
            rfe = RFE(estimator=xgb_for_rfe, n_features_to_select=10)  # Adjust number of features to select
            rfe.fit(X_train, y_train)

            # Get selected features
            selected_features = X.columns[rfe.support_]
            X_train_selected = rfe.transform(X_train)
            X_test_selected = rfe.transform(X_test)

            # Train final XGBoost model on selected features
            xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
            xgb_model.fit(X_train_selected, y_train)

            # Evaluate the model
            y_pred = xgb_model.predict(X_test_selected)
            y_proba = xgb_model.predict_proba(X_test_selected)[:, 1]
            
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall = recall_score(y_test, y_pred, zero_division=0)
            roc_auc = roc_auc_score(y_test, y_proba)
            
            # Confusion matrix to calculate false positive rate
            conf_matrix = confusion_matrix(y_test, y_pred)
            false_positive_rate = conf_matrix[0, 1] / conf_matrix[0].sum() if conf_matrix[0].sum() > 0 else 0

            # End time for processing the dataset
            end_time = time.time()
            runtime = end_time - start_time

            # Save results
            results.append({
                "Dataset Name": filename,
                "Number of Rows": df.shape[0],
                "Number of Columns": df.shape[1],
                "Number of Selected Features": len(selected_features),
                "Selected Features": ', '.join(selected_features),
                "Accuracy": accuracy,
                "Precision": precision,
                "Recall": recall,
                "False Positive Rate": false_positive_rate,
                "ROC AUC": roc_auc,
                "Runtime (seconds)": runtime
            })

        except Exception as e:
            print(f"Error processing {filename}: {e}")
        
        finally:
            # Release memory for this dataset
            del df, X, y, X_resampled, y_resampled, X_train, X_test, y_train, y_test, xgb_for_rfe, rfe, xgb_model
            gc.collect()  # Force garbage collection

# Save results to a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)

print(f"Results saved to {output_csv}")


Processing ds_100K20_.csv...
Error processing ds_100K20_.csv: Unable to allocate 1.00 GiB for an array with shape (3691, 36362) and data type float64
Processing ds_10K18.csv...


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:40:01] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:40:01] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:40:01] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, Us

Processing ds_10K50_rev.csv...


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:40:02] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:40:02] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:40:02] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, Us

Processing ds_11055.csv...


...

C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:40:14] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:40:14] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:40:14] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, Us

Processing ds_129K112.csv...


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:40:36] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:40:38] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\xgboost\core.py:158: UserWarning: [13:40:39] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, Us

Error processing ds_129K112.csv: [13:40:51] C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\common\io.h:320: bad_malloc: Failed to allocate 12531373 bytes.


NameError: name 'xgb_model' is not defined